# Download of the ERA5 data (PECD 4.2)

ERA5 fields of the Pan-European Climate Database (PECD) 4.2 from the Copernicus Climate Data Store (dataset
`sis-energy-pecd`): 10 m wind speed, 2 m temperature and global horizontal irradiance, hourly, 0.25°, 1950–2025, European
domain 18–75° N, 31° W–45° E.

- Needs a free CDS account with the API key in `~/.cdsapirc` and the accepted licence of the dataset.
- One request per year; each year is a zip file of about 2.3 GB with three netCDF files. The files go to `data_external/`.
- Years that are already there are skipped, so the notebook can be run again after an interruption.
- Afterwards run `cd analysis && python organize_raw_data.py`, which links the files into the folders the scripts read.

In [ ]:
import os
import zipfile

import cdsapi

DATASET = "sis-energy-pecd"
VARIABLES = ["10m_wind_speed", "2m_temperature", "surface_solar_radiation_downwards"]
YEARS = range(1950, 2026)
AREA = [75, -31, 18, 45]  # north, west, south, east
OUT = "./data_external"
os.makedirs(OUT, exist_ok=True)


# the netCDF files of one year that are already in the folder
def netcdf_files(year):
    return [f for f in os.listdir(OUT) if f"_S{year}01010000_" in f and f.endswith(".nc") and not f.startswith("._")]

In [ ]:
# one request per year; the zip is written under a temporary name first, so an interrupted download is never mistaken for
# a complete one
client = None
for year in YEARS:
    target = os.path.join(OUT, f"{year}-{year}.zip")
    if len(netcdf_files(year)) == 3 or os.path.exists(target):
        print(f"{year}: already there")
        continue
    client = client or cdsapi.Client()
    request = {"pecd_version": "pecd4_2", "temporal_period": ["historical"], "origin": ["era5_reanalysis"],
               "year": [str(year)], "file_version": ["fv1"], "area": AREA, "variable": VARIABLES,
               "spatial_resolution": ["0_25_degree"]}
    print(f"{year}: requesting -> {target}")
    client.retrieve(DATASET, request).download(target + ".part")
    os.replace(target + ".part", target)

In [ ]:
# unzip every year into the same folder; zips whose files are already unpacked with the right size are skipped
# the netCDF files are compressed already, so the zips can be deleted afterwards to halve the disk space (about 165 GB
# for 1950-2025); set KEEP_ZIPS = False to delete each zip after it has been unpacked
KEEP_ZIPS = True
failed = []
for fname in sorted(os.listdir(OUT)):
    if not fname.endswith(".zip") or fname.startswith("._"):
        continue
    path = os.path.join(OUT, fname)
    try:
        with zipfile.ZipFile(path) as zf:
            done = all(os.path.exists(os.path.join(OUT, m.filename))
                       and os.path.getsize(os.path.join(OUT, m.filename)) == m.file_size for m in zf.infolist())
            if done:
                print(f"{fname}: already unpacked")
            else:
                print(f"{fname}: unpacking")
                zf.extractall(OUT)
    except Exception as e:
        print(f"{fname}: failed ({e})")
        failed.append(fname)
        continue
    if not KEEP_ZIPS:
        os.remove(path)

missing = [y for y in YEARS if len(netcdf_files(y)) != 3]
print(f"failed zips: {failed or 'none'}; years without their three netCDF files: {missing or 'none'}")